# Cleaning the watershed data and turning it into a geoJSON
**NOTE:** I downloaded the watershed_parquet, from one of the TA's github (he shared it with me for help on my project), because for some reason, I am was unable to the donne quebec watershed metadata.**
I kept downloading a .exe file (very weird) and was unable to read any metadata to be able to decode the file. 

In [10]:
import geopandas as gpd
from shapely.ops import polygonize

# importing watershed data 
df = gpd.read_parquet("../site/data/watershed_data.parquet")
print(df.head())

# Importing  quebec region
region = gpd.read_file("../site/data/clean_SDA.geojson")

                         watershed_name  \
0               Northern Arctic Islands   
1  Mackenzie River Delta (Main Channel)   
2                             Porcupine   
3                 Southern Beaufort Sea   
4    Peel and Southwestern Beaufort Sea   

                                            geometry  
0  MULTIPOLYGON (((-84.50922 77.29138, -84.50313 ...  
1  POLYGON ((-133.74303 67.44813, -133.74459 67.4...  
2  POLYGON ((-143.71663 65.88295, -143.72915 65.8...  
3  MULTIPOLYGON (((-128.28288 70.61947, -128.2756...  
4  MULTIPOLYGON (((-143.11826 70.11402, -143.1337...  


## Turning the quebec border into one border polygon and Intersecting with the 

In [11]:

# Joining the lines segements
boundary_lines = region.geometry.union_all()
polygons = list(polygonize(boundary_lines))

quebec_polygon = gpd.GeoDataFrame( geometry=polygons, crs=region.crs,
)

# Saving the quebec polygon
quebec_polygon.to_file("../site/data/quebec_polygon.geojson", driver="GeoJSON", index=False)

# Saving the intersected layers of the watershed
quebec_geom = quebec_polygon.geometry.union_all()
watersheds_in_qc = df.loc[df.geometry.intersects(quebec_geom)].copy()

# Saving it as a file 
watersheds_in_qc.to_file("../site/data/watersheds.geojson", driver="GeoJSON", index=False)